**Подготовка файла 2200.csv на основе данных сводных файлов 2016-2018 годов**

- Автор: Клинкова Екатерина
- Telegram: @ekaklinkova

- `..\Data\{год}\` - папка для хранения созданнного 2200.csv для определенного года
- `..\Data\{год}\Original` - папка для хранения оригинального файла xlsx
- `..\Data\Common\` - папка для хранения итогового результата 2200.csv сводный

Пример: 
- *E:/IrkutskProject/Data/2016/Original/ф33  правл за 2016г Иркутская обл.xlsx*
- *E:/IrkutskProject/Data/2016/2200.csv*

In [4]:
# Загрузка библиотек
import pandas as pd
import numpy as np

In [5]:
# Установка библиотеки для чтения excel. 
!pip install openpyxl
!pip install xlrd

In [6]:
# Проверка версии
# pip show openpyxl

1) Подкотовка датасетов формы 2200 за 2016-2018 года

In [8]:
# Список годов для обработки
years = [2016, 2017, 2018]

# Название оригинального файла
original_names = {
    2016: 'ф33  правл за 2016г Иркутская обл.xlsx',
    2017: 'ф.33 и ф.8 Иркутская область 2017г..xls',
    2018: 'ф.33 и ф.8  Иркутская область  2018.xls'
}
original_url = {}
path = 'E:/IrkutskProject/Data/'

df_2200 = {}

# Путь, где лежит орининальный файл
for year in years:
    original_path = f'{path}{year}/Original/'
    original_url[year] = original_path + original_names[year]
    
    if year == 2016:
        df = pd.read_excel(original_url[year], engine='openpyxl', sheet_name=2)

        # Отфильтруем таблицу 2200
        df_2200[year] = df.head(17)

        # Удаляем пустые столбцы
        df_2200[year] = df_2200[year].drop(df_2200[year].columns[139:218], axis=1)
        df_2200[year] = df_2200[year].drop(df_2200[year].columns[118:138], axis=1)
        df_2200[year] = df_2200[year].drop(df_2200[year].columns[95:117], axis=1)
        df_2200[year] = df_2200[year].drop(df_2200[year].columns[1:94], axis=1)

        # Удаляем лишние строки
        df_2200[year].drop(df_2200[year].index[0:5], inplace=True)

        # Переименовываем колонки
        df_2200[year].columns = ['Наименование показателей', 'Всего', 'Из них детей от 0 до 14 лет', 'Из них подростков от 15 до 17 лет']

    else:
        if year == 2017:
            df = pd.read_excel(original_url[year], engine='xlrd', sheet_name=6)
        elif year == 2018:
            df = pd.read_excel(original_url[year], engine='xlrd', sheet_name=2)

        df_2200[year] = df.head(15)
        df_2200[year] = df_2200[year].drop(df_2200[year].columns[17:19], axis=1)
        df_2200[year] = df_2200[year].drop(df_2200[year].columns[15:16], axis=1)
        df_2200[year] = df_2200[year].drop(df_2200[year].columns[12:13], axis=1)
        df_2200[year] = df_2200[year].drop(df_2200[year].columns[1:11], axis=1)

        df_2200[year].drop(df_2200[year].index[0:3], inplace=True)

        df_2200[year].columns = ['Наименование показателей', 'tmp_column', 'Всего', 'Из них детей от 0 до 14 лет', 'Из них подростков от 15 до 17 лет']
        df_2200[year]['Наименование показателей'] = df_2200[year]['Наименование показателей'].fillna(df_2200[year]['tmp_column'])
        df_2200[year] = df_2200[year].drop(df_2200[year].columns[1:2], axis=1)

    # Обработка заголовков строк
    for row_number in range(1, 3):
        df_2200[year].iloc[row_number, 0] = (df_2200[year].iloc[0, 0] + ' ' + df_2200[year].iloc[row_number, 0]).replace(': ', ' ')

    for row_number in range(3, 5):
        df_2200[year].iloc[row_number, 0] = (df_2200[year].iloc[0, 0] + ' из них с применением ' + df_2200[year].iloc[row_number, 0]).replace(': ', ' ')

    df_2200[year].iloc[5, 0] = (df_2200[year].iloc[0, 0] + ' ' + df_2200[year].iloc[5, 0]).replace(': ', ' ')

    df_2200[year].iloc[6, 0] = df_2200[year].iloc[6, 0].replace('III А', 'III').replace('IIIА', 'III').replace(': ', ' ')

    df_2200[year].iloc[7, 0] = df_2200[year].iloc[7, 0].replace('V  ', 'V ').replace(':', '').replace(' Всего', '') + ' всего'
    
    df_2200[year].iloc[8, 0] = (df_2200[year].iloc[7, 0].replace('V  ', 'V ').replace(':', '').replace(' Всего', '') + ' ' + df_2200[year].iloc[8, 0].strip()).replace('  ', ' ').replace('в VА','VА').replace('в VA','VА')

    df_2200[year].iloc[9, 0] = (df_2200[year].iloc[7, 0].replace('V  ', 'V ').replace(':', '').replace(' Всего', '') + ' в том числе ' + df_2200[year].iloc[9, 0].strip()).replace('  ', ' ')

    
    df_2200[year] = df_2200[year].reset_index(drop=True)
        
    df_2200[year]['Наименование показателей'] = (
        df_2200[year]['Наименование показателей'].astype(str).str.strip()
    )

    # numeric_columns = ['Всего', 'Дети 0-14 лет', 'Подростки 15-17 лет']
    # df_2200[year][numeric_columns] = df_2200[year][numeric_columns].apply(pd.to_numeric, errors='coerce')

    # Выкладываем подготовленные данные в файл
    df_2200[year].to_csv(f'E:/IrkutskProject/Data/{year}/2200.csv')
        

# Путь, где лежит выгруженная страница xlsx в csv формате
# df.to_csv('E:/IrkutskProject/Data/2016/Original/list2.csv')

In [9]:
# df_2200[2016]

In [10]:
# df_2200[2017]

In [11]:
# df_2200[2018]

2) Подготовка сводного датасета 2200.csv

In [13]:
columns = [
    'Наименование показателя',
    'Уточнение',
    'Год',
    'Значение'
]

data = []

df_2200_long = pd.DataFrame(columns=columns)

for year in years:
    for row_number in range(df_2200[year].shape[0]):
        for col_number in range(1, 4):
            new_row = []
            new_row.append(df_2200[year].iloc[row_number, 0].strip())
            new_row.append(df_2200[year].columns[col_number])
            new_row.append(year)
            new_row.append(df_2200[year].iloc[row_number, col_number] if pd.notnull(df_2200[year].iloc[row_number, col_number]) else 0)
            data.append(new_row)
df_2200_long = pd.DataFrame(data, columns=columns)

df_2200_long

,Наименование показателя,Уточнение,Год,Значение
0,Впервые выявлено больных туберкулезом из числа...,Всего,2016,1601
1,Впервые выявлено больных туберкулезом из числа...,Из них детей от 0 до 14 лет,2016,86
2,Впервые выявлено больных туберкулезом из числа...,Из них подростков от 15 до 17 лет,2016,27
3,Впервые выявлено больных туберкулезом из числа...,Всего,2016,97
4,Впервые выявлено больных туберкулезом из числа...,Из них детей от 0 до 14 лет,2016,86
...,...,...,...,...
103,Кроме того умерло больных от туберкулеза посто...,Из них детей от 0 до 14 лет,2018,0
104,Кроме того умерло больных от туберкулеза посто...,Из них подростков от 15 до 17 лет,2018,0
105,Кроме того умерло больных от ВИЧ-инфекции пост...,Всего,2018,1
106,Кроме того умерло больных от ВИЧ-инфекции пост...,Из них детей от 0 до 14 лет,2018,0


In [14]:
df_2200_long.to_csv(f'{path}Common/2200.csv', index=False)